In [42]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import collections

# Загружаем данные
with open('../data/train.jsonl', encoding='utf-8') as f:
    data = [json.loads(l) for l in f if l.strip()]

df = pd.DataFrame(data)
print(f"Total rows: {len(df)}")
print(f"Unique inputs: {df['input'].nunique()}")
print(f"Unique outputs: {df['output'].nunique()}")

# Длины
df['input_len'] = df['input'].str.len()
df['output_len'] = df['output'].str.len()

print(f"\nInput length: min={df['input_len'].min()}, max={df['input_len'].max()}, mean={df['input_len'].mean():.1f}")
print(f"Output length: min={df['output_len'].min()}, max={df['output_len'].max()}, mean={df['output_len'].mean():.1f}")

# Buckets
df['output_bucket'] = pd.cut(df['output_len'], bins=[0, 20, 50, 100, 200, 1000], labels=['<20', '20-50', '50-100', '100-200', '200+'])
print("\nOutput length buckets:")
print(df['output_bucket'].value_counts().sort_index())

# Показываем первые 5
df[['input', 'output']].head()

Total rows: 15208
Unique inputs: 11450
Unique outputs: 2985

Input length: min=2, max=186, mean=18.8
Output length: min=1, max=146, mean=21.5

Output length buckets:
output_bucket
<20        9362
20-50      4850
50-100      948
100-200      48
200+          0
Name: count, dtype: int64


,input,output
0,1/n + 1/m,\frac{1}{n} + \frac{1}{m}
1,1/n+1/m,\frac{1}{n} + \frac{1}{m}
2,1 делить на n + 1 делить на m,\frac{1}{n} + \frac{1}{m}
3,one over n plus one over m,\frac{1}{n} + \frac{1}{m}
4,1/N+1/M,\frac{1}{n} + \frac{1}{m}


In [43]:
print("\nTop 10 outputs:")
print(df['output'].value_counts().head(10))
df_unique = df.drop_duplicates(subset=['input'], keep='first')
print(f"After dedup: {len(df_unique)}")


Top 10 outputs:
output
\lim_{x \to 0} \frac{\sin(x)}{x} = 1                      44
\lim_{x \to 0} \frac{\ln(1+x)}{x} = 1                     39
\lim_{x \to 0} \frac{\tan(x)}{x} = 1                      34
\lim_{x \to 0} \frac{e^{x} - 1}{x} = 1                    30
\lim_{x \to 0} \frac{1 - \cos(x)}{x^{2}} = \frac{1}{2}    30
a^{2} + b^{2} = c^{2}                                     29
\lim_{x \to 0} (1 + x)^{1/x} = e                          25
\lim_{n \to \infty} (1 + \frac{1}{n})^{n} = e             25
\lim_{n \to \infty} (1 + \frac{1}{n})^{n+1} = e           25
\lim_{x \to \infty} (1 + \frac{a}{x})^{x} = e^{a}         25
Name: count, dtype: int64
After dedup: 11450


In [44]:
# Топ-20 outputs
print("Top 20 outputs:")
print(df['output'].value_counts().head(20))

# Топ-20 inputs
print("\nTop 20 inputs:")
print(df['input'].value_counts().head(20))

# Input length distribution
df['input_bucket'] = pd.cut(df['input_len'], bins=[0, 20, 50, 100, 200], labels=['<20', '20-50', '50-100', '100-200'])
print("\nInput length buckets:")
print(df['input_bucket'].value_counts().sort_index())

Top 20 outputs:
output
\lim_{x \to 0} \frac{\sin(x)}{x} = 1                                      44
\lim_{x \to 0} \frac{\ln(1+x)}{x} = 1                                     39
\lim_{x \to 0} \frac{\tan(x)}{x} = 1                                      34
\lim_{x \to 0} \frac{e^{x} - 1}{x} = 1                                    30
\lim_{x \to 0} \frac{1 - \cos(x)}{x^{2}} = \frac{1}{2}                    30
a^{2} + b^{2} = c^{2}                                                     29
\lim_{x \to 0} (1 + x)^{1/x} = e                                          25
\lim_{n \to \infty} (1 + \frac{1}{n})^{n} = e                             25
\lim_{n \to \infty} (1 + \frac{1}{n})^{n+1} = e                           25
\lim_{x \to \infty} (1 + \frac{a}{x})^{x} = e^{a}                         25
\lim_{x \to \infty} (1 + \frac{a}{x})^{bx} = e^{ab}                       25
\lim_{x \to 0} (1 + ax)^{b/x} = e^{ab}                                    25
\lim_{x \to 0} (1 + x)^{a/x} = e^{a}                 

In [45]:
# Оставляем по 5 примеров на каждый уникальный output
df_balanced = df.groupby('output').head(5).reset_index(drop=True)

In [46]:
def categorize(output):
    if '\\lim' in output: return 'limit'
    if '\\int' in output: return 'integral'
    if '\\sum' in output: return 'sum'
    if '\\frac{d}{dx}' in output or "f'" in output: return 'derivative'
    if '\\sqrt' in output: return 'sqrt'
    if '\\log' in output or '\\ln' in output: return 'log'
    if '\\sin' in output or '\\cos' in output or '\\tan' in output: return 'trig'
    if '\\frac' in output: return 'fraction'
    if '=' in output: return 'equation'
    if '\\alpha' in output or '\\beta' in output or '\\gamma' in output or '\\theta' in output: return 'greek'
    if '^' in output: return 'power'
    return 'simple'

df['category'] = df['output'].apply(categorize)

# Проверка
print(df['category'].value_counts())

category
fraction      2580
equation      2080
power         1860
simple        1779
limit         1466
trig          1116
sqrt           943
sum            805
integral       691
derivative     687
log            651
greek          550
Name: count, dtype: int64


In [47]:
MAX_PER_CATEGORY = 1500

# Группируем и сэмплируем вручную
sampled_dfs = []
for cat, group in df.groupby('category'):
    if len(group) > MAX_PER_CATEGORY:
        sampled = group.sample(MAX_PER_CATEGORY, random_state=42)
    else:
        sampled = group
    sampled_dfs.append(sampled)

df_balanced = pd.concat(sampled_dfs, ignore_index=True)

print(f"After downsampling: {len(df_balanced)}")
print(f"Columns: {df_balanced.columns.tolist()}")
print("\nCategory distribution:")
print(df_balanced['category'].value_counts())
print("\nPercentages:")
print((df_balanced['category'].value_counts(normalize=True) * 100).round(1))

print(f"After downsampling: {len(df_balanced)}")
print(f"Columns: {df_balanced.columns.tolist()}")
print("\nCategory distribution:")
print(df_balanced['category'].value_counts())
print("\nPercentages:")
print((df_balanced['category'].value_counts(normalize=True) * 100).round(1))

After downsampling: 12909
Columns: ['input', 'output', 'input_len', 'output_len', 'output_bucket', 'input_bucket', 'category']

Category distribution:
category
equation      1500
fraction      1500
power         1500
simple        1500
limit         1466
trig          1116
sqrt           943
sum            805
integral       691
derivative     687
log            651
greek          550
Name: count, dtype: int64

Percentages:
category
equation      11.6
fraction      11.6
power         11.6
simple        11.6
limit         11.4
trig           8.6
sqrt           7.3
sum            6.2
integral       5.4
derivative     5.3
log            5.0
greek          4.3
Name: proportion, dtype: float64
After downsampling: 12909
Columns: ['input', 'output', 'input_len', 'output_len', 'output_bucket', 'input_bucket', 'category']

Category distribution:
category
equation      1500
fraction      1500
power         1500
simple        1500
limit         1466
trig          1116
sqrt           943
sum      

In [49]:
unique_inputs = df_balanced['input'].nunique()
print(f"Unique inputs: {unique_inputs} / {len(df_balanced)}")
print(f"Duplicates: {len(df_balanced) - unique_inputs}")

Unique inputs: 9945 / 12909
Duplicates: 2964


In [50]:
# Дедупликация
df_unique = df_balanced.drop_duplicates(subset=['input'], keep='first').reset_index(drop=True)

print(f"After dedup: {len(df_unique)}")
print(f"Removed: {len(df_balanced) - len(df_unique)}")
print("\nCategory distribution:")
print(df_unique['category'].value_counts())
print("\nPercentages:")
print((df_unique['category'].value_counts(normalize=True) * 100).round(1))

After dedup: 9945
Removed: 2964

Category distribution:
category
power         1290
fraction      1284
simple        1264
equation      1207
trig           775
sqrt           726
limit          683
sum            657
derivative     614
log            541
integral       505
greek          399
Name: count, dtype: int64

Percentages:
category
power         13.0
fraction      12.9
simple        12.7
equation      12.1
trig           7.8
sqrt           7.3
limit          6.9
sum            6.6
derivative     6.2
log            5.4
integral       5.1
greek          4.0
Name: proportion, dtype: float64


In [51]:
with open('../data/train_balanced.jsonl', 'w', encoding='utf-8') as f:
    for _, row in df_unique.iterrows():
        f.write(json.dumps({'input': row['input'], 'output': row['output']}, ensure_ascii=False) + '\n')

print(f"Saved: {len(df_unique)} rows")
print(f"Path: ../data/train_balanced.jsonl")

Saved: 9945 rows
Path: ../data/train_balanced.jsonl


In [52]:
# Загружаем текущий датасет
with open('../data/train_balanced.jsonl', encoding='utf-8') as f:
    data = [json.loads(l) for l in f if l.strip()]

df = pd.DataFrame(data)

print(f"Total rows: {len(df)}")
print(f"Unique inputs: {df['input'].nunique()}")
print(f"Unique outputs: {df['output'].nunique()}")

# Дубликаты
dup_inputs = df['input'].duplicated().sum()
dup_pairs = df.duplicated(subset=['input', 'output']).sum()
print(f"\nDuplicate inputs: {dup_inputs}")
print(f"Duplicate (input, output) pairs: {dup_pairs}")

Total rows: 9945
Unique inputs: 9945
Unique outputs: 2764

Duplicate inputs: 0
Duplicate (input, output) pairs: 0


In [53]:
# Длины
df['input_len'] = df['input'].str.len()
df['output_len'] = df['output'].str.len()

# Категории
def categorize(output):
    if '\\lim' in output: return 'limit'
    if '\\int' in output: return 'integral'
    if '\\sum' in output: return 'sum'
    if '\\frac{d}{dx}' in output or "f'" in output: return 'derivative'
    if '\\sqrt' in output: return 'sqrt'
    if '\\log' in output or '\\ln' in output: return 'log'
    if '\\sin' in output or '\\cos' in output or '\\tan' in output: return 'trig'
    if '\\frac' in output: return 'fraction'
    if '=' in output: return 'equation'
    if '\\alpha' in output or '\\beta' in output or '\\gamma' in output or '\\theta' in output: return 'greek'
    if '^' in output: return 'power'
    return 'simple'

df['category'] = df['output'].apply(categorize)

# Категории
print("=== CATEGORIES ===")
cat_counts = df['category'].value_counts()
cat_pct = df['category'].value_counts(normalize=True) * 100
for cat in cat_counts.index:
    print(f"  {cat:12s}: {cat_counts[cat]:5d} ({cat_pct[cat]:.1f}%)")

# Длины
print(f"\n=== LENGTHS ===")
print(f"Input  len: min={df['input_len'].min()}, max={df['input_len'].max()}, mean={df['input_len'].mean():.1f}")
print(f"Output len: min={df['output_len'].min()}, max={df['output_len'].max()}, mean={df['output_len'].mean():.1f}")

# Buckets по output
df['out_bucket'] = pd.cut(df['output_len'], bins=[0, 15, 30, 60, 120, 300], labels=['<15', '15-30', '30-60', '60-120', '120+'])
print(f"\nOutput length buckets:")
print(df['out_bucket'].value_counts().sort_index())

# Outputs per output
print(f"\n=== DUPLICATES PER OUTPUT ===")
output_counts = df['output'].value_counts()
print(f"Max inputs per output: {output_counts.max()}")
print(f"Min inputs per output: {output_counts.min()}")
print(f"Mean inputs per output: {output_counts.mean():.2f}")
print(f"\nTop 10 most repeated outputs:")
for out, cnt in output_counts.head(10).items():
    print(f"  {cnt:3d}× {out[:70]!r}")

=== CATEGORIES ===
  power       :  1290 (13.0%)
  fraction    :  1284 (12.9%)
  simple      :  1264 (12.7%)
  equation    :  1207 (12.1%)
  trig        :   775 (7.8%)
  sqrt        :   726 (7.3%)
  limit       :   683 (6.9%)
  sum         :   657 (6.6%)
  derivative  :   614 (6.2%)
  log         :   541 (5.4%)
  integral    :   505 (5.1%)
  greek       :   399 (4.0%)

=== LENGTHS ===
Input  len: min=2, max=186, mean=19.1
Output len: min=1, max=146, mean=21.4

Output length buckets:
out_bucket
<15       4712
15-30     3015
30-60     1959
60-120     249
120+        10
Name: count, dtype: int64

=== DUPLICATES PER OUTPUT ===
Max inputs per output: 17
Min inputs per output: 1
Mean inputs per output: 3.60

Top 10 most repeated outputs:
   17× '\\lim_{x \\to 0} \\frac{\\sin(x)}{x} = 1'
   17× '\\lim_{x \\to 0} \\frac{\\ln(1+x)}{x} = 1'
   16× '\\lim_{x \\to 0} \\frac{\\tan(x)}{x} = 1'
   11× '\\lim_{x \\to 0} \\frac{e^{x} - 1}{x} = 1'
   10× 'e^{i\\pi} + 1 = 0'
   10× '\\int_{0}^{\\infty} e

In [54]:
# Сколько outputs повторяются 10+ раз?
output_counts = df['output'].value_counts()

for n in [1, 2, 3, 5, 10, 15]:
    count = (output_counts == n).sum()
    print(f"Outputs with {n:2d} inputs: {count}")

print(f"\nOutputs with 5+ inputs: {(output_counts >= 5).sum()}")
print(f"Outputs with 10+ inputs: {(output_counts >= 10).sum()}")

# Сколько строк занимают outputs с 10+ inputs?
top_outputs = output_counts[output_counts >= 10].index
top_rows = df[df['output'].isin(top_outputs)]
print(f"\nRows with top outputs (10+ inputs): {len(top_rows)} ({len(top_rows)/len(df)*100:.1f}%)")

Outputs with  1 inputs: 661
Outputs with  2 inputs: 183
Outputs with  3 inputs: 331
Outputs with  5 inputs: 934
Outputs with 10 inputs: 6
Outputs with 15 inputs: 0

Outputs with 5+ inputs: 1148
Outputs with 10+ inputs: 10

Rows with top outputs (10+ inputs): 121 (1.2%)


In [55]:
# Короткие (<15 символов)
short = df[df['output_len'] < 15]
print(f"Short rows: {len(short)}")
print(f"Unique short inputs: {short['input'].nunique()}")
print(f"Unique short outputs: {short['output'].nunique()}")

# Распределение
short_outputs = short['output'].value_counts()
print(f"\nShort outputs with 5+ inputs: {(short_outputs >= 5).sum()}")
print(f"Short outputs with 1 input: {(short_outputs == 1).sum()}")

Short rows: 4272
Unique short inputs: 4272
Unique short outputs: 1378

Short outputs with 5+ inputs: 428
Short outputs with 1 input: 500


In [56]:
from collections import defaultdict

with open('../data/train_balanced.jsonl', encoding='utf-8') as f:
    data = [json.loads(l) for l in f if l.strip()]

# Группировка по output
by_output = defaultdict(list)
for d in data:
    by_output[d['output']].append(d)

# Оставляем не более 3 inputs на output
MAX_INPUTS = 3
balanced = []
for out, items in by_output.items():
    balanced.extend(items[:MAX_INPUTS])

print(f"Original: {len(data)}")
print(f"After max-3: {len(balanced)}")

# Сохраняем
with open('../data/train_max3.jsonl', 'w', encoding='utf-8') as f:
    for d in balanced:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')

print("Saved to data/train_max3.jsonl")

Original: 9945
After max-3: 6787
Saved to data/train_max3.jsonl


In [57]:

with open('../data/train_max3.jsonl', encoding='utf-8') as f:
    data = [json.loads(l) for l in f if l.strip()]

df3 = pd.DataFrame(data)

df3['input_len'] = df3['input'].str.len()
df3['output_len'] = df3['output'].str.len()

print(f"Total: {len(df3)}")
print(f"Unique inputs: {df3['input'].nunique()}")
print(f"Unique outputs: {df3['output'].nunique()}")

# Категории
def categorize(output):
    if '\\lim' in output: return 'limit'
    if '\\int' in output: return 'integral'
    if '\\sum' in output: return 'sum'
    if '\\frac{d}{dx}' in output or "f'" in output: return 'derivative'
    if '\\sqrt' in output: return 'sqrt'
    if '\\log' in output or '\\ln' in output: return 'log'
    if '\\sin' in output or '\\cos' in output or '\\tan' in output: return 'trig'
    if '\\frac' in output: return 'fraction'
    if '=' in output: return 'equation'
    if '\\alpha' in output or '\\beta' in output or '\\gamma' in output or '\\theta' in output: return 'greek'
    if '^' in output: return 'power'
    return 'simple'

df3['category'] = df3['output'].apply(categorize)

print("\n=== CATEGORIES ===")
cat_counts = df3['category'].value_counts()
for cat in cat_counts.index:
    pct = cat_counts[cat] / len(df3) * 100
    print(f"  {cat:12s}: {cat_counts[cat]:5d} ({pct:.1f}%)")

# Длины
print(f"\n=== LENGTHS ===")
print(f"Input  mean: {df3['input_len'].mean():.1f}")
print(f"Output mean: {df3['output_len'].mean():.1f}")

df3['out_bucket'] = pd.cut(df3['output_len'], bins=[0, 15, 30, 60, 120], labels=['<15', '15-30', '30-60', '60-120'])
print(f"\nOutput buckets:")
print(df3['out_bucket'].value_counts().sort_index())

# Inputs per output
output_counts = df3['output'].value_counts()
print(f"\n=== INPUTS PER OUTPUT ===")
print(f"Max: {output_counts.max()}")
print(f"Min: {output_counts.min()}")
print(f"Mean: {output_counts.mean():.2f}")

Total: 6787
Unique inputs: 6787
Unique outputs: 2764

=== CATEGORIES ===
  fraction    :  1132 (16.7%)
  power       :   959 (14.1%)
  simple      :   911 (13.4%)
  equation    :   901 (13.3%)
  trig        :   481 (7.1%)
  sqrt        :   406 (6.0%)
  limit       :   395 (5.8%)
  sum         :   386 (5.7%)
  derivative  :   360 (5.3%)
  log         :   331 (4.9%)
  integral    :   291 (4.3%)
  greek       :   234 (3.4%)

=== LENGTHS ===
Input  mean: 17.7
Output mean: 20.7

Output buckets:
out_bucket
<15       3379
15-30     2022
30-60     1213
60-120     165
Name: count, dtype: int64

=== INPUTS PER OUTPUT ===
Max: 3
Min: 1
Mean: 2.46


In [58]:
# Длины в токенах (приблизительно)
df3['input_tokens'] = df3['input'].str.len() / 3
df3['output_tokens'] = df3['output'].str.len() / 3

# Сколько токенов в каждом бакете?
buckets = {
    'short (<15)': df3[df3['output_len'] < 15],
    'medium (15-30)': df3[(df3['output_len'] >= 15) & (df3['output_len'] < 30)],
    'long (30-60)': df3[(df3['output_len'] >= 30) & (df3['output_len'] < 60)],
    'very_long (60+)': df3[df3['output_len'] >= 60],
}

total_tokens = 0
for name, group in buckets.items():
    tokens = group['output_tokens'].sum()
    total_tokens += tokens
    print(f"{name:20s}: {len(group):5d} rows, {tokens:8.0f} tokens")

print(f"\nTotal tokens: {total_tokens:.0f}")

# Процент токенов
print(f"\n=== TOKEN DISTRIBUTION ===")
for name, group in buckets.items():
    tokens = group['output_tokens'].sum()
    print(f"{name:20s}: {tokens/total_tokens*100:.1f}% of tokens")

short (<15)         :  3059 rows,     9612 tokens
medium (15-30)      :  2297 rows,    15577 tokens
long (30-60)        :  1252 rows,    17044 tokens
very_long (60+)     :   179 rows,     4658 tokens

Total tokens: 46891

=== TOKEN DISTRIBUTION ===
short (<15)         : 20.5% of tokens
medium (15-30)      : 33.2% of tokens
long (30-60)        : 36.3% of tokens
very_long (60+)     : 9.9% of tokens


In [ ]:
# ============================================================
# Pre-training checklist
# ============================================================

import os
import json
import inspect

print("=== CHECKLIST ===\n")

# 1. Files
files_to_check = [
    'src/model.py',
    'src/loss.py',
    'src/math_tokenizer.py',
    'src/dataset.py',
    'data/train_max3.jsonl',
]
for f in files_to_check:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    print(f"{status} {f}")

# 2. Model updated?
from src.model import SRFTM
print(f"\ngreedy_decode params: {inspect.signature(SRFTM.greedy_decode)}")
print(f"has beam_search: {hasattr(SRFTM, 'beam_search')}")

# 3. Data size
with open('data/train_max3.jsonl', encoding='utf-8') as f:
    data = [json.loads(l) for l in f if l.strip()]
print(f"\nTotal rows: {len(data)}")

# 4. Data split
print(f"train_split.jsonl exists: {os.path.exists('data/train_split.jsonl')}")
print(f"val_split.jsonl exists: {os.path.exists('data/val_split.jsonl')}")

# 6. Tokenizer
from src.math_tokenizer import MathTokenizer
tokenizer = MathTokenizer()
print(f"\nVocab size: {len(tokenizer)}")

# 7. Loss
from src.loss import LengthWeightedLoss
print(f"LengthWeightedLoss imported: ✅")

=== CHECKLIST ===

❌ src/model.py
❌ src/loss.py
❌ src/math_tokenizer.py
❌ src/dataset.py
❌ data/train_max3.jsonl


ModuleNotFoundError: No module named 'src.model'